## XG Boost Classifier on Titanic dataset to predict survival

### Load Dataset

In [29]:
import seaborn as sns
import pandas as pd

df = sns.load_dataset("titanic")

print(df.tail())
print(df.shape)

     survived  pclass     sex   age  sibsp  parch   fare embarked   class  \
886         0       2    male  27.0      0      0  13.00        S  Second   
887         1       1  female  19.0      0      0  30.00        S   First   
888         0       3  female   NaN      1      2  23.45        S   Third   
889         1       1    male  26.0      0      0  30.00        C   First   
890         0       3    male  32.0      0      0   7.75        Q   Third   

       who  adult_male deck  embark_town alive  alone  
886    man        True  NaN  Southampton    no   True  
887  woman       False    B  Southampton   yes   True  
888  woman       False  NaN  Southampton    no  False  
889    man        True    C    Cherbourg   yes   True  
890    man        True  NaN   Queenstown    no   True  
(891, 15)


### Handling Missing Values

In [30]:
print(df.isnull().sum())

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64


In [31]:
df = df.drop("deck",axis=1)
print(df.columns)

Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked', 'class', 'who', 'adult_male', 'embark_town', 'alive',
       'alone'],
      dtype='object')


In [32]:
df['age'] = df['age'].fillna(df['age'].median())
df['embark_town'] = df['embark_town'].fillna(df['embark_town'].mode())
df['embarked'] = df['embarked'].fillna(df['embarked'].mode())

In [33]:
X = df.drop(['survived','who','alive'],axis=1)
y = df['survived']

In [34]:
X = pd.get_dummies(X,columns=['sex','class','embarked','embark_town','alone'],drop_first=True)

### Train test split

In [35]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [36]:
print(X_train.info())

<class 'pandas.core.frame.DataFrame'>
Index: 712 entries, 331 to 102
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   pclass                   712 non-null    int64  
 1   age                      712 non-null    float64
 2   sibsp                    712 non-null    int64  
 3   parch                    712 non-null    int64  
 4   fare                     712 non-null    float64
 5   adult_male               712 non-null    bool   
 6   sex_male                 712 non-null    bool   
 7   class_Second             712 non-null    bool   
 8   class_Third              712 non-null    bool   
 9   embarked_Q               712 non-null    bool   
 10  embarked_S               712 non-null    bool   
 11  embark_town_Queenstown   712 non-null    bool   
 12  embark_town_Southampton  712 non-null    bool   
 13  alone_True               712 non-null    bool   
dtypes: bool(9), float64(2), int64

### Feature scaling

In [37]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### Model training

In [38]:
from xgboost import XGBClassifier
model = XGBClassifier(n_estimators=5,max_depth=5,learning_rate=0.7)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)


### Evaluation

In [39]:
from sklearn.metrics import (accuracy_score,confusion_matrix,precision_score,recall_score,f1_score,roc_auc_score)

y_prob = model.predict_proba(X_test)[:,1]
print(f"Accuracy: {accuracy_score(y_test,y_pred)}")
print(f"Confusion Matrix:\n {confusion_matrix(y_test,y_pred)}")
print(f"Precision: {precision_score(y_test,y_pred)}")
print(f"Recall: {recall_score(y_test,y_pred)}")
print(f"F1 score: {f1_score(y_test,y_pred)}")
print(f"ROC AUC score: {roc_auc_score(y_test,y_prob)}")

Accuracy: 0.8491620111731844
Confusion Matrix:
 [[95 10]
 [17 57]]
Precision: 0.8507462686567164
Recall: 0.7702702702702703
F1 score: 0.8085106382978723
ROC AUC score: 0.8901544401544401


### Interpret Coefficients

In [40]:
print(model.get_booster().get_score(importance_type='weight'))

{'f0': 4.0, 'f1': 28.0, 'f2': 3.0, 'f3': 2.0, 'f4': 32.0, 'f5': 2.0, 'f6': 2.0, 'f7': 2.0, 'f10': 4.0}
